# Kaya Identity와 LMDI 분해분석

## 국가 CO₂ 배출 변화의 요인별 기여도 계산

비율, 변화율과 로그비의 의미를 작은 수치 예제로 확인한 뒤 Kaya 항등식을 구성합니다. 이어서 두 시점 사이의 배출량 변화를 LMDI로 분해하고, 네 요인의 기여도 합이 실제 변화량과 일치하는지 검증합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 두 값의 비율, 퍼센트 변화율, 로그비가 변화를 서로 다른 방식으로 표현한다는 점을 설명합니다.
- Kaya 항등식의 인구·1인당 GDP·에너지집약도·탄소집약도를 정의하고 곱의 단위가 CO₂ 배출량으로 복원되는지 확인합니다.
- Kaya 항등식이 배출량의 구조를 표현하고, additive LMDI가 두 시점의 총변화를 요인별 기여도로 나눈다는 차이를 설명합니다.
- 로그평균 가중치로 기여도를 계산하고, 기여도의 합이 실제 변화량과 일치하는지 검증합니다.
- 요인별 기여도를 인과효과로 과장하지 않고 국가·기간 선택과 해석 범위를 함께 기록합니다.


## 이번 교시에서 사용할 데이터

### Kaya–LMDI 수업용 국가 snapshot

| 항목 | 내용 |
|---|---|
| 구성 | Germany, Japan, South Korea, United States |
| 기간과 규모 | 1990~2024년, 140행 |
| 한 행의 의미 | 한 국가의 한 연도 CO₂·경제·에너지 관측 |
| 원자료 열 | `co2_tonnes`, `co2_tonnes_per_capita`, `gdp_per_capita_intl2021`, `primary_energy_twh` |
| 수업에서 계산할 값 | 인구, 총 GDP, 에너지집약도, 탄소집약도 |

이 파일은 Global Carbon Budget/OWID의 CO₂, World Bank/OWID의 1인당 GDP, U.S. EIA의 1차에너지 소비량을 같은 국가와 기간에 맞춘 검증 snapshot이며 원 논문의 분석자료가 아닙니다. EIA 에너지는 TJ를 TWh로 변환했고, 독일 1990년은 EIA 동독·서독 계열을 합산했습니다. 인구는 총배출량과 1인당 배출량의 비로 복원하고, 총 GDP와 두 집약도는 해당 값을 이용해 계산합니다.

강의에서는 South Korea의 2000~2022년을 사용합니다. 적용 실습에서는 국가 또는 시작·종료연도를 하나만 변경해 결과 차이를 비교합니다. LMDI의 로그비를 계산하려면 선택한 두 시점의 모든 요인이 양수여야 합니다.


## 논문의 방법을 Python으로 재현한다는 뜻

방법론 재현은 논문의 문장을 요약하는 작업과 다릅니다. 논문에 제시된 식을 데이터 열과 계산 순서로 번역하고, 중간값과 검산 결과를 남기는 작업입니다.

이번 교시에는 Ang (2005)의 additive LMDI 절차를 축소된 국가–연도 자료에 적용합니다.

`논문의 식 → 필요한 열 → 두 시점 입력 → 요인 변화 → 기여도 → closure → 해석 범위`

원 논문의 표본과 결과를 그대로 복제하는 것이 아니라, **논문 방법을 Python에서 구현하고 검증하는 방법**을 익히는 예입니다.

출처: [Ang (2005), *The LMDI Approach to Decomposition Analysis: A Practical Guide*](https://doi.org/10.1016/j.enpol.2003.10.010)


## Kaya 항등식을 계산 가능한 네 열로 번역하기

Kaya 항등식은 에너지 관련 CO₂ 배출량을 네 요인의 곱으로 표현합니다.

$$
C=P\times\frac{G}{P}\times\frac{E}{G}\times\frac{C}{E}
$$

| Python 열 | 식 | 의미 |
|---|---|---|
| `population` | $P$ | 인구 |
| `gdp_per_capita` | $G/P$ | 1인당 경제활동 수준 |
| `energy_intensity` | $E/G$ | GDP 한 단위당 에너지 사용량 |
| `carbon_intensity` | $C/E$ | 에너지 한 단위당 CO₂ 배출량 |

이 식은 회귀계수를 추정하는 모형이 아니라 분자와 분모가 약분되어 $C$가 되는 정의 관계입니다. 분석용 자료에서는 네 열의 단위가 서로 호환되는지 먼저 확인합니다.


## 항등식의 곱과 단위를 코드로 검산하기

두 시점의 네 요인을 곱한 값이 기록된 CO₂와 일치해야 다음 분해 계산으로 넘어갈 수 있습니다.

$$
\text{Kaya product}
=P\times(G/P)\times(E/G)\times(C/E)
$$

값이 일치하지 않으면 단위 변환, 분모 정의, 자료 범위, 반올림을 먼저 점검합니다. 항등식 검산을 통과하지 않은 상태에서 LMDI 기여도를 계산하면 그럴듯한 숫자가 나오더라도 해석할 수 없습니다.

다음 실행 셀에서는 두 시점의 네 열을 직접 곱하고, 기록값과의 차이를 표로 확인합니다.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

endpoints = pd.DataFrame(
    {
        "period": ["시작", "종료"],
        "population": [50.0, 52.0],
        "gdp_per_capita": [40.0, 44.0],
        "energy_intensity": [0.150, 0.130],
        "carbon_intensity": [0.250, 0.220],
    }
)
factor_columns = ["population", "gdp_per_capita", "energy_intensity", "carbon_intensity"]
endpoints["kaya_product"] = endpoints[factor_columns].prod(axis=1)
endpoints["recorded_co2"] = [75.0000, 65.4368]
endpoints["difference"] = endpoints["kaya_product"] - endpoints["recorded_co2"]

print("[두 시점의 Kaya 입력과 곱 검산]")
display(endpoints.round(6))
passed = np.allclose(endpoints["kaya_product"], endpoints["recorded_co2"], atol=1e-9)
print(f"Kaya product와 기록 CO₂ 일치: {passed}")
print(f"최대 절대차: {endpoints['difference'].abs().max():.10f}")
print("검산이 실패하면 LMDI보다 먼저 단위와 열 정의를 확인합니다.")


## 두 시점의 비율과 로그변화를 나란히 읽기

LMDI 계산에 들어가기 전에 각 요인이 어느 방향으로 얼마나 바뀌었는지 확인합니다.

$$
r_i=\frac{X_{i,T}}{X_{i,0}}, \qquad
\Delta\ln X_i=\ln(r_i)
$$

- 비율이 1보다 크면 증가, 1보다 작으면 감소입니다.
- 로그변화가 양수면 증가, 음수면 감소입니다.
- 로그는 0이나 음수에 적용할 수 없으므로 두 시점 입력이 모두 양수인지 검사합니다.

다음 실행 셀에서는 네 요인의 시작값·종료값·비율·로그변화를 한 표와 막대그래프로 연결합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

factors = pd.DataFrame(
    {
        "factor": ["인구", "1인당 GDP", "에너지집약도", "탄소집약도"],
        "start": [50.0, 40.0, 0.150, 0.250],
        "end": [52.0, 44.0, 0.130, 0.220],
    }
)
if (factors[["start", "end"]] <= 0).any().any():
    raise ValueError("로그변화를 계산하려면 두 시점 값이 모두 양수여야 합니다.")
factors["ratio"] = factors["end"] / factors["start"]
factors["percent_change"] = (factors["ratio"] - 1) * 100
factors["log_change"] = np.log(factors["ratio"])
factors["direction"] = np.where(factors["ratio"] >= 1, "증가", "감소")

print("[요인별 두 시점 변화]")
display(factors.round({"start": 3, "end": 3, "ratio": 3, "percent_change": 1, "log_change": 3}))
print("ratio 1과 log-change 0이 변화 방향의 기준입니다.")

colors = np.where(factors["log_change"] >= 0, "#E45756", "#4C78A8")
fig, ax = plt.subplots(figsize=(8, 3.5))
factor_plot_labels = ["Population", "GDP per capita", "Energy intensity", "Carbon intensity"]
ax.bar(factor_plot_labels, factors["log_change"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Log change by factor", xlabel="", ylabel="ln(end / start)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


## 로그평균이 총변화와 요인 변화를 연결한다

additive LMDI에서 각 요인의 기여도는 두 시점 CO₂의 로그평균과 그 요인의 로그변화를 곱해 계산합니다.

$$
L(C_T,C_0)=\frac{C_T-C_0}{\ln C_T-\ln C_0}
$$

$$
\Delta C_i=L(C_T,C_0)\ln\left(\frac{X_{i,T}}{X_{i,0}}\right)
$$

로그평균은 시작점과 종료점의 정보를 함께 반영하는 가중치입니다. 이번 구현에서는 두 시점의 CO₂와 네 요인이 모두 양수인지 먼저 검사하며, 0값에는 별도의 처리 규칙이 필요합니다.


## additive LMDI 기여도와 closure를 함께 계산하기

네 기여도는 원래 CO₂와 같은 단위로 표시됩니다. 양수는 총변화를 증가시키는 방향, 음수는 감소시키는 방향을 뜻합니다.

$$
\Delta C_{observed}=C_T-C_0
$$

$$
\Delta C_{population}+\Delta C_{affluence}
+\Delta C_{energy}+\Delta C_{carbon}
\approx \Delta C_{observed}
$$

마지막 등식의 차이를 **closure error**로 기록합니다. 이는 계산 조각을 다시 더했을 때 원래 변화와 일치하는지 확인하는 구현 검산이며, 각 요인이 인과적 원인임을 증명하는 검정은 아닙니다.

다음 실행 셀에서는 기여도 표, 막대그래프, 관측 변화와 합계의 차이를 함께 확인합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

factor_names = ["인구", "1인당 GDP", "에너지집약도", "탄소집약도"]
start = np.array([50.0, 40.0, 0.150, 0.250])
end = np.array([52.0, 44.0, 0.130, 0.220])
co2_start = float(np.prod(start))
co2_end = float(np.prod(end))

if np.any(start <= 0) or np.any(end <= 0):
    raise ValueError("LMDI 입력은 모두 양수여야 합니다.")
log_mean = (co2_end - co2_start) / (np.log(co2_end) - np.log(co2_start))
contributions = log_mean * np.log(end / start)
observed_change = co2_end - co2_start
contribution_sum = float(contributions.sum())
closure_error = contribution_sum - observed_change

result = pd.DataFrame(
    {
        "factor": factor_names,
        "ratio": end / start,
        "contribution": contributions,
        "direction": np.where(contributions >= 0, "증가 방향", "감소 방향"),
    }
)
closure = pd.DataFrame(
    {
        "검산 항목": ["관측 CO₂ 변화", "네 기여도 합", "closure error"],
        "값": [observed_change, contribution_sum, closure_error],
    }
)

print("[additive LMDI 기여도]")
display(result.round(4))
print("\n[closure 검산]")
display(closure.round(10))
print(f"closure 통과: {np.isclose(contribution_sum, observed_change, atol=1e-9)}")

colors = np.where(result["contribution"] >= 0, "#E45756", "#4C78A8")
fig, ax = plt.subplots(figsize=(8, 3.6))
factor_plot_labels = ["Population", "GDP per capita", "Energy intensity", "Carbon intensity"]
ax.bar(factor_plot_labels, result["contribution"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Additive contribution by factor", xlabel="", ylabel="CO₂ change (example unit)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


## 재현 코드가 통과해야 할 검증 조건

논문의 식을 코드로 옮긴 뒤에는 결과값보다 먼저 다음 조건을 검사합니다.

1. 시작·종료 시점의 네 요인이 모두 양수인가?
2. 각 시점의 Kaya product가 기록된 CO₂와 일치하는가?
3. 요인 비율의 방향이 원자료와 일치하는가?
4. 네 기여도의 합이 관측 CO₂ 변화와 일치하는가?
5. 국가, 기간, 단위와 분해 방식이 결과표에 남아 있는가?

이 검산을 함수와 표로 남기면 국가나 기간을 바꾸어도 같은 분석 절차를 반복할 수 있습니다.


## 방법 재현의 결과와 해석 범위

이번 결과로 보고할 수 있는 것은 선택한 국가와 기간에서 **총변화가 네 요인에 산술적으로 얼마씩 배분됐는가**입니다.

다음과 같은 인과 문장은 이 분석만으로 뒷받침되지 않습니다.

- 특정 정책 때문에 배출량이 정확히 몇 톤 감소했다.
- 가장 큰 양의 기여도가 가장 중요한 정책 원인이다.
- 한 국가·한 기간의 결과가 다른 국가와 기간에도 그대로 적용된다.

결과 문장에는 `표본 + 기간 + additive LMDI + 기여 방향과 단위 + closure + 해석 한계`를 함께 제시합니다. 이것이 논문 방법을 계산 가능한 Python 분석으로 옮기는 최소 기록입니다.


## 실행 환경과 입력자료

| 재현 조건 | 확인할 출력 |
|---|---|
| 입력자료 | 실제 파일 경로와 파일명 |
| 자료 구조 | 데이터 소개에 제시한 예상 `shape`와 필수 열 |
| 실행환경 | Python과 주요 라이브러리 버전 |

다음 셀은 분석에 사용한 파일과 실행환경을 기록합니다. 파일 경로 또는 `shape`가 강의 기준과 다르면 입력 파일과 버전을 확인한 뒤 분석을 진행합니다.

> **실행 전 확인:** 이번 분석의 입력 파일과 예상 행 수는 무엇인가요?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 변화비·퍼센트·로그비 계산

세 가지 종료값을 시작값 100과 비교합니다. 비율·퍼센트 변화와 로그 변화비의 부호가 어떻게 연결되는지 확인하세요.


In [ ]:
ratio_example = pd.DataFrame({"start": [100.0, 100.0, 100.0], "end": [110.0, 100.0, 80.0]})
ratio_example["end_start_ratio"] = ratio_example["end"] / ratio_example["start"]
ratio_example["percent_change"] = (ratio_example["end_start_ratio"] - 1) * 100
ratio_example["log_ratio"] = np.log(ratio_example["end_start_ratio"])
ratio_example["direction"] = np.select(
    [ratio_example["log_ratio"] > 0, ratio_example["log_ratio"] < 0],
    ["increase", "decrease"],
    default="no change",
)
display(ratio_example.round(4))
print("로그 변화비의 부호는 증가·변화 없음·감소 방향과 일치합니다.")


## 1. 입력자료와 파생요인 검증

LMDI 계산에 사용할 원자료와 파생요인을 구분합니다.

| 요인 | 계산 | 단위 예시 | 적용 조건 |
|---|---|---|---|
| 인구 | 총 CO₂ ÷ 1인당 CO₂ | persons | 양수 |
| 1인당 GDP | 공개지표 | intl-$ / person | 양수 |
| 에너지집약도 | 에너지 ÷ 총 GDP | TWh / intl-$ | 양수 |
| 탄소집약도 | CO₂ ÷ 에너지 | tonnes / TWh | 양수 |

파생요인을 계산한 뒤 네 요인의 곱이 원자료의 총 CO₂와 일치하는지 확인합니다. 국가–연도 key와 단위는 계산 전후에 유지합니다.

> **확인 질문:** 에너지집약도와 탄소집약도는 각각 무엇을 분모로 사용합니까?


In [ ]:
LMDI_PATH = find_course_data("methods/kaya_lmdi_sample.csv")
kaya = pd.read_csv(LMDI_PATH)
analysis_name = "Kaya-LMDI South Korea"

print("shape:", kaya.shape)
print("countries:", sorted(kaya["code"].unique()))
print("year range:", kaya["year"].min(), kaya["year"].max())
display(kaya.head(3))


### 출력 해석: 분해에 들어가는 원자료

한 국가의 시작·종료 연도를 나란히 놓고 CO₂·인구 관련 값·GDP·에너지의 변화를 먼저 확인합니다.


In [ ]:
input_view = kaya.loc[
    kaya["code"].eq("KOR") & kaya["year"].isin([2000, 2022]),
    ["year", "co2_tonnes", "co2_tonnes_per_capita", "gdp_per_capita_intl2021", "primary_energy_twh"],
]
print("LMDI는 아래 두 시점의 총변화를 요인별로 나눕니다.")
display(input_view)


## 2. Additive LMDI의 입력과 출력

| 구분 | 내용 |
|---|---|
| 입력 | 국가, 시작연도, 종료연도, 네 개의 양수 Kaya 요인 |
| 가중치 | 시작·종료 CO₂의 로그평균 |
| 계산 | 요인의 로그 변화비 × 로그평균 가중치 |
| 출력 | 네 기여도, 관측 CO₂ 변화량, closure residual |

기여도는 CO₂ 변화량과 같은 단위로 계산한 뒤 Mt CO₂로 표시합니다. 0 또는 음수 요인이 발견되면 현재 함수의 적용 조건을 벗어난 것으로 처리합니다.

> **확인 질문:** 네 기여도의 합은 어떤 관측값과 일치해야 합니까?


In [ ]:
def log_mean(x1: float, x0: float) -> float:
    if np.isclose(x1, x0):
        return float(x1)
    return float((x1 - x0) / (np.log(x1) - np.log(x0)))

def additive_lmdi(frame: pd.DataFrame, code: str, start_year: int, end_year: int):
    country = frame.loc[frame["code"].eq(code)].sort_values("year").copy()
    endpoints = country.loc[country["year"].isin([start_year, end_year])].set_index("year")
    if set(endpoints.index) != {start_year, end_year}:
        raise ValueError("선택한 시작·종료 연도가 데이터에 모두 있어야 합니다.")

    endpoints["population"] = endpoints["co2_tonnes"] / endpoints["co2_tonnes_per_capita"]
    endpoints["gdp_total"] = endpoints["population"] * endpoints["gdp_per_capita_intl2021"]
    endpoints["energy_intensity"] = endpoints["primary_energy_twh"] / endpoints["gdp_total"]
    endpoints["carbon_intensity"] = endpoints["co2_tonnes"] / endpoints["primary_energy_twh"]

    factor_columns = {
        "population": "population effect",
        "gdp_per_capita_intl2021": "affluence effect",
        "energy_intensity": "energy-intensity effect",
        "carbon_intensity": "carbon-intensity effect",
    }
    c0 = float(endpoints.loc[start_year, "co2_tonnes"])
    c1 = float(endpoints.loc[end_year, "co2_tonnes"])
    weight = log_mean(c1, c0)
    effects = []
    for column, label in factor_columns.items():
        f0 = float(endpoints.loc[start_year, column])
        f1 = float(endpoints.loc[end_year, column])
        effects.append({"effect": label, "contribution_tonnes": weight * np.log(f1 / f0)})
    result = pd.DataFrame(effects)
    return endpoints.reset_index(), result, c1 - c0


## 3. 국가와 분석기간 설정

LMDI 결과는 선택한 국가와 두 endpoint에 대응합니다.

| 설정 | 강의 시연 | 분석에 미치는 영향 |
|---|---|---|
| 국가 | `KOR` | 경제·산업·에너지 구조 |
| 시작연도 | 2000 | 요인 변화비의 기준점 |
| 종료연도 | 2022 | 총변화량과 기여도 규모 |

두 시점 분해에서는 중간연도의 경로가 직접 사용되지 않습니다. 국가나 기간을 바꾸면 총변화와 요인별 기여도도 달라질 수 있으므로 설정을 결과표에 명시합니다.

> **출력 예측:** `end/start ratio < 1`은 해당 요인이 어떤 방향으로 변했음을 뜻합니까?


In [ ]:
COUNTRY_CODE = "KOR"
START_YEAR = 2000
END_YEAR = 2022

endpoints, contributions, observed_change = additive_lmdi(
    kaya, COUNTRY_CODE, START_YEAR, END_YEAR
)
result_table = contributions.assign(contribution_mt=lambda d: d["contribution_tonnes"] / 1_000_000)
display(endpoints[["year", "co2_tonnes", "population", "gdp_total", "primary_energy_twh"]])
display(result_table[["effect", "contribution_mt"]])
print("observed CO2 change (Mt):", observed_change / 1_000_000)
print("sum of effects (Mt):", result_table["contribution_mt"].sum())


### 출력 해석: 시작값·종료값·변화비

각 Kaya 요인이 증가했는지 감소했는지 기여도를 보기 전에 확인합니다.


In [ ]:
factor_columns_view = [
    "population", "gdp_per_capita_intl2021", "energy_intensity", "carbon_intensity"
]
factor_view = endpoints.set_index("year")[factor_columns_view].T
factor_view.columns = [f"{START_YEAR}", f"{END_YEAR}"]
factor_view["end/start ratio"] = factor_view[f"{END_YEAR}"] / factor_view[f"{START_YEAR}"]
print("ratio > 1이면 증가, ratio < 1이면 감소입니다.")
display(factor_view)

closure_view = pd.DataFrame({
    "quantity": ["observed CO2 change", "sum of four effects"],
    "Mt CO2": [observed_change / 1_000_000, result_table["contribution_mt"].sum()],
})
print("완전분해에서는 아래 두 값이 일치해야 합니다.")
display(closure_view)


## 4. 기여도와 Closure 검증

기여도 표와 그림은 다음 순서로 해석합니다.

$$
\Delta C_{observed}=\Delta C_P+\Delta C_A+\Delta C_E+\Delta C_F
$$

1. 시작·종료 CO₂와 네 요인의 값·단위를 확인합니다.
2. 요인 변화비와 기여도 부호의 방향을 비교합니다.
3. 관측 변화량과 기여도 합계의 차이를 계산합니다.
4. 모든 기여도를 Mt CO₂ 단위로 비교합니다.

가장 큰 막대는 선택한 국가·기간·요인 정의에서 가장 큰 산술적 기여입니다. 정책의 독립적 인과효과를 평가하려면 별도의 식별전략이 필요합니다.

> **확인 질문:** closure residual이 0에 가깝다는 결과와 정책효과가 확인되었다는 주장은 어떻게 다릅니까?


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
colors = ["#c44e52" if value > 0 else "#4c72b0" for value in result_table["contribution_mt"]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(result_table["effect"], result_table["contribution_mt"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title=f"Additive LMDI | {COUNTRY_CODE}, {START_YEAR}–{END_YEAR}", xlabel="Contribution to CO2 change (Mt)")
plt.tight_layout()
plt.show()

closure_error = float(result_table["contribution_tonnes"].sum() - observed_change)
checks = pd.Series({
    "two endpoints": len(endpoints) == 2,
    "positive factor values": (endpoints.select_dtypes("number").drop(columns="year") > 0).all().all(),
    "four effects": len(result_table) == 4,
    "complete decomposition": abs(closure_error) < 1e-4,
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("LMDI 분해 검증: PASS")


## ChatGPT 저장 응답 검증: LMDI 결과를 과장 없이 설명하기

아래 프롬프트는 계산된 네 기여도와 완전분해 오차를 이용해 **산술적 기여와 인과효과를 구분한 결과문장**을 점검합니다.

답변에서 네 Kaya 요인의 정의·단위, 기여도의 부호와 합을 확인합니다. 정책효과나 원인으로 단정한 문장은 현재 분석으로 뒷받침되지 않으므로 수정합니다.


In [ ]:
review_prompt = f"""
다음 LMDI 결과의 해석을 점검해 주세요.

- 국가와 기간: {COUNTRY_CODE}, {START_YEAR}~{END_YEAR}
- 관측 CO₂ 변화: {observed_change / 1_000_000:.3f} Mt
- 기여도:\n{result_table[['effect', 'contribution_mt']].to_string(index=False)}
- closure error: {closure_error:.6f} tonnes

아래 순서로 답해 주세요.
1. 네 기여도의 부호와 합에 근거한 관찰
2. 완전분해 확인 결과
3. 과장된 인과표현이 없는 결과문장 2개
4. 국가·기간 선택에 대한 다음 민감성 검사 1개

제약: 새로운 숫자를 만들거나 기여도를 정책의 인과효과로 표현하지 마세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. 네 요인의 정의와 단위를 입력표에서 다시 확인합니다.
2. 기여도 합과 관측 변화량의 차이가 허용오차 안인지 확인합니다.
3. `원인`, `효과`, `정책 때문에` 같은 표현은 인과설계가 없으므로 사용하지 않습니다.


## 해석 범위

LMDI 결과는 선택한 국가와 두 시점 사이의 배출 변화량을 Kaya 요인별 산술적 기여도로 분해합니다.

기여도는 정책·기술·인구의 독립적인 인과효과가 아닙니다. 시작·종료 연도와 지표의 출처가 달라지면 결과도 달라질 수 있으므로, 국가·기간·자료 출처와 closure 검증을 함께 보고합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

Kaya 항등식으로 배출량의 네 요인을 정의하고, LMDI로 두 시점의 변화를 네 기여도로 나눈 뒤 완전분해를 확인합니다.

### 적용 순서

1. `03_kaya_lmdi_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `03_kaya_lmdi_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 시작값에서 종료값으로의 변화를 비율, 퍼센트 변화율, 로그비로 표현할 때 각 값은 무엇을 다르게 보여줍니까?
- Kaya 항등식의 네 요인을 곱하면 왜 CO₂ 배출량이 되며, LMDI는 이 항등식에 어떤 질문을 추가합니까?
- LMDI 기여도의 합이 실제 변화량과 일치한다는 사실이 요인의 인과효과까지 증명합니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
